# Project 3: Email/SMS Spam Detection
**Level:** Beginner to intermediate  
**Goal:** Classify short messages as spam or ham (not spam) using TF-IDF and machine learning.

This notebook uses a small **built-in teaching dataset** so it can run offline. It is only a demonstration, not a production-quality spam filter. Replace it with a larger, properly licensed dataset for serious evaluation.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, classification_report, ConfusionMatrixDisplay)

examples = [
("ham","Hey, are we still meeting for lunch today?"),
("ham","Please send me the notes from the lecture."),
("ham","Your appointment is confirmed for tomorrow morning."),
("ham","Can you call me when you arrive?"),
("ham","The project meeting starts at 3 pm."),
("ham","Thanks for your help with the assignment."),
("ham","I will be ten minutes late."),
("ham","Remember to bring your notebook to class."),
("ham","Dinner is ready whenever you get home."),
("ham","The report has been uploaded to the shared folder."),
("ham","Could you review this draft before Friday?"),
("ham","Happy birthday! Hope you have a great day."),
("ham","Your package has been delivered at reception."),
("ham","The train is delayed by about fifteen minutes."),
("ham","I have attached the updated schedule."),
("ham","Let's discuss the experiment results tomorrow."),
("spam","Congratulations! You have won a free prize. Claim now!"),
("spam","URGENT: claim your cash reward by clicking this link."),
("spam","You are selected for a guaranteed bonus. Reply WIN."),
("spam","Limited offer! Buy now and save big."),
("spam","Free entry in our prize draw. Text YES today."),
("spam","You have been approved for an exclusive cash loan."),
("spam","Act now to claim your gift card."),
("spam","Winner! Call this number to collect your reward."),
("spam","Earn money fast from home. Sign up now."),
("spam","Your account won a reward. Verify details immediately."),
("spam","Special discount just for you, purchase today."),
("spam","Claim your free voucher before it expires."),
("spam","Get a cash bonus now. Limited time offer."),
("spam","You have won a holiday. Click to claim."),
("spam","Lowest price guaranteed. Buy today."),
("spam","Congratulations, claim your free subscription now.")
]
df = pd.DataFrame(examples, columns=["label","message"])
display(df.sample(8, random_state=42))
print(df["label"].value_counts())

## 1. Split messages into training and test sets

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df["message"], df["label"], test_size=0.25, random_state=42, stratify=df["label"]
)

## 2. Train TF-IDF models

In [ ]:
models = {
    "Multinomial Naive Bayes": Pipeline([
        ("tfidf", TfidfVectorizer(lowercase=True, ngram_range=(1,2), stop_words="english")),
        ("clf", MultinomialNB())
    ]),
    "Logistic Regression": Pipeline([
        ("tfidf", TfidfVectorizer(lowercase=True, ngram_range=(1,2), stop_words="english")),
        ("clf", LogisticRegression(max_iter=1000, random_state=42))
    ])
}
rows = []
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    rows.append({"Model":name,
                 "Accuracy":accuracy_score(y_test,pred),
                 "Precision (spam)":precision_score(y_test,pred,pos_label="spam",zero_division=0),
                 "Recall (spam)":recall_score(y_test,pred,pos_label="spam",zero_division=0),
                 "F1 (spam)":f1_score(y_test,pred,pos_label="spam",zero_division=0)})
metrics = pd.DataFrame(rows).sort_values("F1 (spam)", ascending=False)
metrics.round(3)

## 3. Evaluate the best model

In [ ]:
best_name = metrics.iloc[0]["Model"]
best_model = models[best_name]
pred = best_model.predict(X_test)
print("Selected model:", best_name)
print(classification_report(y_test, pred, zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_test, pred, labels=["ham","spam"])
plt.title("Spam detection confusion matrix")
plt.tight_layout(); plt.show()

## 4. Test new messages

In [ ]:
new_messages = [
    "Congratulations, claim your free reward today!",
    "Hi, could you send the meeting notes this afternoon?",
    "Urgent limited offer, click now to win cash."
]
predictions = best_model.predict(new_messages)
for message, label in zip(new_messages, predictions):
    print(f"{label.upper():4} | {message}")

## 5. Limitations and improvements
- The dataset is tiny and deliberately simplified; test metrics are unstable and not representative of real-world spam.
- Real messages may contain URLs, misspellings, multilingual text, and changing spam tactics.
- Use a larger licensed corpus, cross-validation, and a held-out test set.
- Consider the cost of false positives: legitimate messages should not be silently discarded.